# 使用SNN完成MNIST数据集的分类
- 通过**ANN转SNN**的方式，使用ANN训练MNIST数据集，然后将训练好的ANN转换为SNN进行测试。

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

## ANN

### 1. 模型

In [2]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=512, num_classes=10):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten the input
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

### 2. 训练

In [ ]:
def train(epochs=5):
    # 设备设置
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    # 迭代次数
    # 数据预处理
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    # 加载MNIST数据集
    train_dataset = datasets.MNIST(root='./MNIST', train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root='./MNIST', train=False, transform=transform, download=True)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    # 模型
    model = MLP().to(device)
    # 损失
    criterion = nn.CrossEntropyLoss()
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    # 训练循环
    print('Starting training...')
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            # 统计
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            if (batch_idx + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}, Accuracy: {100.0 * correct / total:.2f}%')
        
        # 测试
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        print(f'[Test] Loss: {test_loss/len(test_loader):.4f}, Test Accuracy: {100.0 * correct / total:.2f}%')

    # 保存模型
    if not os.path.exists('./models'):
        os.makedirs('./models')
    torch.save(model.state_dict(), f'./models/mlp_epoch_{epochs}.pth')
    print('Model saved.')


In [7]:
train(5)

Using device: cuda
Starting training...
Epoch [1/5], Batch [100/938], Loss: 0.3110, Accuracy: 81.11%
Epoch [1/5], Batch [200/938], Loss: 0.3688, Accuracy: 85.55%
Epoch [1/5], Batch [300/938], Loss: 0.2665, Accuracy: 87.88%
Epoch [1/5], Batch [400/938], Loss: 0.0716, Accuracy: 89.16%
Epoch [1/5], Batch [500/938], Loss: 0.1938, Accuracy: 90.16%
Epoch [1/5], Batch [600/938], Loss: 0.1850, Accuracy: 90.93%
Epoch [1/5], Batch [700/938], Loss: 0.2552, Accuracy: 91.48%
Epoch [1/5], Batch [800/938], Loss: 0.3591, Accuracy: 91.85%
Epoch [1/5], Batch [900/938], Loss: 0.0712, Accuracy: 92.21%
[Test] Loss: 0.1196, Test Accuracy: 96.23%
Model saved.
Epoch [2/5], Batch [100/938], Loss: 0.0567, Accuracy: 95.67%
Epoch [2/5], Batch [200/938], Loss: 0.1508, Accuracy: 95.72%
Epoch [2/5], Batch [300/938], Loss: 0.1049, Accuracy: 95.78%
Epoch [2/5], Batch [400/938], Loss: 0.1234, Accuracy: 95.90%
Epoch [2/5], Batch [500/938], Loss: 0.1901, Accuracy: 95.92%
Epoch [2/5], Batch [600/938], Loss: 0.0801, Accura

### 3. 推理

## 转SNN

In [8]:
# 使用一个小框架
from layers import *

In [ ]:
class MLP_SNN(nn.Module):
    def __init__(self, input_size=784, hidden_size=512, num_classes=10, T=6):
        super(MLP_SNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        # self.dropout = nn.Dropout(0.3)
        # SNN相关层
        self.T = T
        self.act = LIFSpike()
        self.fc1_s = tdLayer(self.fc1)
        self.fc2_s = tdLayer(self.fc2)
        self.fc3_s = tdLayer(self.fc3)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten the input
        # 对输入数据增加时间维度
        x = add_dimention(x, self.T)
        x = self.act(self.fc1_s(x))   # 将ReLU替换为LIF激活函数，经过self.fc1_s后数据还是浮点数，需要经过LIF激活函数(self.act)转换为脉冲信号0或1
        # x = self.dropout(x)
        x = self.act(self.fc2_s(x))
        # x = self.dropout(x)
        x = self.fc3_s(x)

        # 输出维度取平均
        x = x.mean(dim=1)
        return x

In [ ]:
# SNN训练
def train_SNN(epochs=5):
    # 设备设置
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    # 迭代次数
    # 数据预处理
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    # 加载MNIST数据集
    train_dataset = datasets.MNIST(root='./MNIST', train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root='./MNIST', train=False, transform=transform, download=True)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    # 模型
    model = MLP_SNN().to(device)
    # 损失
    criterion = nn.CrossEntropyLoss()
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    # 训练循环
    print('Starting training...')
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            # 统计
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            if (batch_idx + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}, Accuracy: {100.0 * correct / total:.2f}%')
        
        # 测试
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        print(f'[Test] Loss: {test_loss/len(test_loader):.4f}, Test Accuracy: {100.0 * correct / total:.2f}%')

    # 保存模型
    if not os.path.exists('./models'):
        os.makedirs('./models')
    torch.save(model.state_dict(), f'./models/mlp_epoch_{epochs}.pth')
    print('Model saved.')


In [15]:
train_SNN()

Using device: cuda
Starting training...
Epoch [1/5], Batch [100/938], Loss: 0.5108, Accuracy: 83.69%
Epoch [1/5], Batch [200/938], Loss: 0.1977, Accuracy: 87.84%
Epoch [1/5], Batch [300/938], Loss: 0.2463, Accuracy: 89.91%
Epoch [1/5], Batch [400/938], Loss: 0.1393, Accuracy: 90.95%
Epoch [1/5], Batch [500/938], Loss: 0.1880, Accuracy: 91.70%
Epoch [1/5], Batch [600/938], Loss: 0.1114, Accuracy: 92.28%
Epoch [1/5], Batch [700/938], Loss: 0.0440, Accuracy: 92.75%
Epoch [1/5], Batch [800/938], Loss: 0.2347, Accuracy: 93.06%
Epoch [1/5], Batch [900/938], Loss: 0.0721, Accuracy: 93.38%
[Test] Loss: 0.1279, Test Accuracy: 95.84%
Model saved.
Epoch [2/5], Batch [100/938], Loss: 0.0571, Accuracy: 96.55%
Epoch [2/5], Batch [200/938], Loss: 0.1811, Accuracy: 96.48%
Epoch [2/5], Batch [300/938], Loss: 0.1190, Accuracy: 96.46%
Epoch [2/5], Batch [400/938], Loss: 0.0163, Accuracy: 96.46%
Epoch [2/5], Batch [500/938], Loss: 0.1250, Accuracy: 96.49%
Epoch [2/5], Batch [600/938], Loss: 0.2307, Accura

- **但是这里SNN的输入数据不是脉冲**，采用的是**直接编码**的方式，即第一层直接输入实数，由第一层神经元（如 IF 神经元）积攒电位并产生脉冲。

- 可见SNN比ANN的性能、准确率要稍差